# Fase 1.3 — Dataset analítico: predictores en t0, fugas y exclusiones

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Con el objetivo ya construido, armo la tabla que van a usar el EDA y los modelos: una fila por crédito con **solo información disponible en la fecha de solicitud (t0)**. Esto corrige tres problemas de la v1:

- **Percentiles transductivos.** La v1 calculaba percentiles (monto, salario, tamaño del empleador) con toda la muestra, prueba incluida. Aquí dejo las variables en su escala original; cualquier percentil se calculará dentro del pliegue en la Fase 3.
- **Historia del cliente.** La "peor mora previa" de la v1 solo miraba créditos ya cerrados. Aquí la calculo con la mora observada **hasta el día anterior a t0** en todos los créditos previos, cerrados o activos. Es más información y sigue siendo pasado.
- **Fugas y exclusiones.** Dejo por escrito, variable por variable, qué entra, qué no y por qué. El género sale por completo.

El resultado se guarda en `_privado/dataset_analitico_v2.parquet` (con filas, fuera de git). Aquí solo muestro agregados.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import numpy as np
import pandas as pd

from tesis_v2 import config, io_datos, mora, objetivo
from tesis_v2.io_datos import fecha, llave, numero

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_rows", 80, "display.max_colwidth", 70)

R = objetivo.cargar_y_reconstruir(guardar_series=True)
series = R["series"]
obj = io_datos.leer_parquet(config.PRIVADO / "objetivo_v2.parquet")
cred, sol = R["tablas"]["credito"], R["tablas"]["solicitud"]
print(f"Créditos en el objetivo v2: {len(obj)}")

Créditos en el objetivo v2: 1161


## 1. Auditoría de fugas, columna por columna

Antes de construir variables, clasifico cada columna de `bd_credito` y `bd_solicitud` según **cuándo se conoce**. Solo pueden entrar como predictores las que se fijan en la originación. Las que se actualizan con la vida del crédito (estado, categoría, saldos) quedan vetadas aunque parezcan inocentes.

In [2]:
auditoria = pd.DataFrame([
    ("bd_credito", "valortotal, cantidadcuotas, cuotapactada, tasa", "Originación", "Entra", "Términos pactados del crédito"),
    ("bd_credito", "cnfrecaudo", "Originación", "Entra", "Modalidad de recaudo pactada (empresarial, mensual, quincenal)"),
    ("bd_credito", "tipo_aval, aval, cnfdescuento_aval", "Originación", "Entra", "Garantía pactada"),
    ("bd_credito", "valor_seguro, porcentaje_seguro, estudio_credito, cnfgastos", "Originación", "Entra (derivadas)", "Costos asociados al crédito"),
    ("bd_credito", "tipocredito_id", "Originación", "Filtro + marca de prenda", "Define la población (libranza) y la marca con prenda"),
    ("bd_credito", "empresa_id", "Originación", "Solo derivadas", "El id del empleador no entra; se usa su historia previa a t0"),
    ("bd_credito", "fechavencimientocredito", "Originación", "No (redundante)", "Equivale al plazo, que se calcula del plan"),
    ("bd_credito", "estado", "Posterior a t0", "Vetada (fuga)", "Estado actual del crédito"),
    ("bd_credito", "categoria_credito", "Posterior a t0", "Vetada (fuga)", "Cambia a REFINANCIADO cuando se refinancia"),
    ("bd_credito", "en_refinanciacion", "Posterior a t0", "Vetada (fuga)", "APROBADA coincide con refinanciaciones futuras (notebook 01)"),
    ("bd_credito", "categoria_inicial", "Originación", "No (constante)", "Vale NUEVO en el 100 % de los créditos"),
    ("bd_credito", "gasto_perfil, gasto_adicional, puntos_seguridad_seguro, tasa_anual_seguro", "Originación", "No", "Casi constantes o sin documentación"),
    ("bd_solicitud", "tipo_solicitud", "Originación", "Entra", "Crédito nuevo o refinanciación"),
    ("bd_solicitud", "montosolicitado", "Originación", "Entra (razón)", "Monto aprobado / solicitado"),
    ("bd_solicitud", "estado, fue_modificado, cantidad_modificaciones", "Proceso", "No", "Estado de la solicitud; fue_modificado es constante"),
    ("bd_solicitud", "cuotarecaudada", "Ambigua", "No", "Sin documentación clara de cuándo se fija"),
    ("bd_saldo, bd_saldomesames, bd_pagosempresas, bd_avales", "todas", "Posterior a t0", "Vetada (fuga)", "Describen el desempeño del crédito"),
    ("bd_cliente", "fechanacimiento", "Originación", "Solo edad en t0", "La fecha se descarta tras calcular la edad"),
    ("bd_cliente", "genero", "Originación", "Excluida por decisión", "Decisión 6: el género no entra al modelado"),
    ("bd_cliente", "departamentos_id, ciudades_id", "Originación", "Excluida", "Minimización de datos y riesgo de proxy socioeconómico"),
    ("bd_cliente", "nombres, documento, correo, teléfono, dirección", "—", "Nunca se leen", "Datos personales (Ley 1581)"),
    ("Buró / PILA (fase 2D v1)", "score, obligaciones, moras, salario, PILA", "Consulta en la originación", "Entra si la consulta es anterior al desembolso", "Disponible desde dic-2025"),
    ("Huella digital", "todas", "Originación (desde ago-2026)", "Excluida del modelo principal", "Sin créditos maduros y la versión de la v1 era sintética"),
], columns=["fuente", "columnas", "momento", "decision", "razon"])
io_datos.guardar_resultado(auditoria, "fase1", "auditoria_fugas")
auditoria

,fuente,columnas,momento,decision,razon
0,bd_credito,"valortotal, cantidadcuotas, cuotapactada, tasa",Originación,Entra,Términos pactados del crédito
1,bd_credito,cnfrecaudo,Originación,Entra,"Modalidad de recaudo pactada (empresarial, mensual, quincenal)"
2,bd_credito,"tipo_aval, aval, cnfdescuento_aval",Originación,Entra,Garantía pactada
3,bd_credito,"valor_seguro, porcentaje_seguro, estudio_credito, cnfgastos",Originación,Entra (derivadas),Costos asociados al crédito
4,bd_credito,tipocredito_id,Originación,Filtro + marca de prenda,Define la población (libranza) y la marca con prenda
5,bd_credito,empresa_id,Originación,Solo derivadas,El id del empleador no entra; se usa su historia previa a t0
6,bd_credito,fechavencimientocredito,Originación,No (redundante),"Equivale al plazo, que se calcula del plan"
7,bd_credito,estado,Posterior a t0,Vetada (fuga),Estado actual del crédito
8,bd_credito,categoria_credito,Posterior a t0,Vetada (fuga),Cambia a REFINANCIADO cuando se refinancia
9,bd_credito,en_refinanciacion,Posterior a t0,Vetada (fuga),APROBADA coincide con refinanciaciones futuras (notebook 01)


## 2. Términos del crédito y del cliente en t0

In [3]:
base = R["base"].set_index("cid")
c = cred.assign(cid=llave(cred["id"])).set_index("cid").loc[base.index]
d = pd.DataFrame(index=base.index)
d["monto"] = numero(c["valortotal"])
d["n_cuotas"] = numero(c["cantidadcuotas"])
d["tasa_mensual"] = numero(c["tasa"])
plan_fin = R["plan"].groupby("cid")["fecha"].max()
d["plazo_meses"] = (plan_fin.reindex(d.index) - base["t_ref"]).dt.days / 30.44
d["cuota_mensual"] = numero(c["cuotapactada"]) * d["n_cuotas"] / d["plazo_meses"].where(d["plazo_meses"] > 0.5)
d["recaudo"] = c["cnfrecaudo"].str.lower()
d["tiene_aval"] = c["tipo_aval"].notna().astype(int)
d["aval_pct"] = numero(c["aval"])
d["cobra_seguro"] = (numero(c["valor_seguro"]).fillna(0) > 0).astype(int)
d["cobra_estudio"] = (numero(c["estudio_credito"]).fillna(0) > 0).astype(int)
d["cobra_4x1000"] = c["cnfgastos"].str.contains("4MIL", na=False).astype(int)
d["con_prenda"] = llave(c["tipocredito_id"]).isin([2, 4]).astype(int)

s = sol.assign(sid=llave(sol["id"])).set_index("sid")
sid = base["sid"]
d["tipo_solicitud"] = s["tipo_solicitud"].reindex(sid).values
d["tipo_solicitud"] = d["tipo_solicitud"].fillna("SIN_SOLICITUD").str.lower()
d["es_refinanciacion"] = d["tipo_solicitud"].str.startswith("refinanciacion").astype(int)
monto_sol = numero(s["montosolicitado"]).reindex(sid).values
d["monto_sobre_solicitado"] = d["monto"] / pd.Series(monto_sol, index=d.index).where(lambda x: x > 0)

nac = io_datos.leer_parquet(config.EXPORT_FINTECH / "bd_cliente.parquet", ["id", "fechanacimiento"])
f_nac = pd.Series(fecha(nac["fechanacimiento"]).values, index=llave(nac["id"]))
d["edad_t0"] = (base["t0"] - base["cli"].map(f_nac)).dt.days / 365.25
d.loc[(d["edad_t0"] < 18) | (d["edad_t0"] > 80), "edad_t0"] = np.nan
del nac, f_nac
d.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
monto,1161.0,5931180.099053,9403227.561292,200000.0,1500000.0,2694520.0,6348300.0,135229592.0
n_cuotas,1161.0,18.492679,15.435689,1.0,10.0,14.0,24.0,270.0
tasa_mensual,1161.0,1.800738,0.347986,0.0,1.57,1.78,1.98,3.0
plazo_meses,1161.0,18.595002,15.439378,0.361367,10.019711,13.69908,23.9159,270.00657
cuota_mensual,1160.0,529073.478718,1672363.115682,29940.983607,150849.057696,243081.145613,406653.229974,29280758.915714
tiene_aval,1161.0,0.596038,0.490902,0.0,0.0,1.0,1.0,1.0
aval_pct,692.0,4.598266,2.184778,2.0,2.0,5.0,5.0,12.0
cobra_seguro,1161.0,0.264427,0.441218,0.0,0.0,0.0,1.0,1.0
cobra_estudio,1161.0,0.340224,0.473989,0.0,0.0,0.0,1.0,1.0
cobra_4x1000,1161.0,0.241171,0.427978,0.0,0.0,0.0,0.0,1.0


Las variables monetarias quedan en pesos y la tasa en % mensual. La razón monto/solicitado solo tiene dato en 153 créditos: el campo `montosolicitado` casi nunca se diligencia. La dejo en el diccionario por transparencia, pero no es candidata al modelo. Los máximos de monto y de número de cuotas corresponden a créditos de libre inversión con prenda, que quedan fuera de la población primaria.

## 3. Historia del cliente y del empleador antes de t0

Para cada crédito miro los créditos **anteriores del mismo grupo de cliente** (desembolsados antes de t0) y calculo:

- cuántos tuvo y cuántos seguían activos en t0;
- el peor DPD que alcanzaron **hasta el día anterior a t0**, con la serie diaria reconstruida;
- hace cuántos meses tomó su primer crédito;
- cómo terminó el último crédito cerrado antes de t0.

Del empleador solo uso cuántos créditos de FINTECH había originado antes de t0, una medida de antigüedad de la relación con el convenio. No uso su tasa de mora histórica: con tan pocos eventos sería una variable muy ruidosa y con riesgo de fuga si no se fecha con cuidado.

In [4]:
o = obj.set_index("cid")
info = pd.DataFrame({"g": o["grupo_cliente"], "t0": o["t0"], "t_ref": o["t_ref"], "fin": o["fin_obs_A"],
                     "cierre": o["cierre_v2"], "emp": base["emp"]})
pares = info.reset_index().merge(info.reset_index(), on="g", suffixes=("", "_p"))
pares = pares[(pares["cid"] != pares["cid_p"]) & (pares["t_ref_p"] < pares["t0"])]
pares["dpd_previo"] = [mora.max_dpd_hasta(series, cp, t - pd.Timedelta(days=1))
                       for cp, t in zip(pares["cid_p"], pares["t0"])]
pares["activo_en_t0"] = pares["fin_p"] >= pares["t0"]
h = pares.groupby("cid").agg(n_creditos_previos=("cid_p", "size"), n_activos_t0=("activo_en_t0", "sum"),
                             peor_mora_previa=("dpd_previo", "max"), primer_credito=("t_ref_p", "min"))
cerr = pares[~pares["activo_en_t0"]].sort_values("fin_p").groupby("cid")["cierre_p"].last()
d = d.join(h)
d["n_creditos_previos"] = d["n_creditos_previos"].fillna(0).astype(int)
d["n_activos_t0"] = d["n_activos_t0"].fillna(0).astype(int)
d["es_recurrente"] = (d["n_creditos_previos"] > 0).astype(int)
d["meses_desde_primer_credito"] = ((base["t0"] - d["primer_credito"]).dt.days / 30.44)
mapa = {"A_TERMINO": "a_termino", "PREPAGO": "prepago", "REFINANCIACION": "refinanciacion",
        "REFINANCIACION_NO_ENLAZADA": "refinanciacion", "CON_SALDO_CLIENTE_RECURRENTE": "con_saldo"}
d["cierre_previo"] = np.where(d["n_creditos_previos"] == 0, "sin_previo",
                              cerr.reindex(d.index).map(mapa).fillna("otro_o_activo"))
d = d.drop(columns="primer_credito")

orden = info.sort_values("t0")
d["emp_creditos_previos"] = orden.groupby("emp").cumcount().reindex(d.index).where(info["emp"].notna())
d["sin_empleador"] = info["emp"].isna().astype(int)
d[["n_creditos_previos", "n_activos_t0", "peor_mora_previa", "meses_desde_primer_credito", "emp_creditos_previos"]].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
n_creditos_previos,1161.0,1.1,1.5,0.0,0.0,0.0,2.0,9.0
n_activos_t0,1161.0,0.4,0.5,0.0,0.0,0.0,1.0,3.0
peor_mora_previa,570.0,19.0,45.8,0.0,1.0,8.0,25.0,611.0
meses_desde_primer_credito,570.0,16.9,11.5,0.4,7.4,13.7,23.4,53.1
emp_creditos_previos,1116.0,119.2,130.3,0.0,17.0,64.0,180.2,459.0


**Control anti-fuga de la historia.** Si la construcción es correcta, ningún crédito previo usado en la historia puede haberse desembolsado en t0 o después, y el DPD previo se mide antes de t0. Lo verifico de forma automática.

In [5]:
assert (pares["t_ref_p"] < pares["t0"]).all()
print("Pares crédito–crédito previo usados:", len(pares))
print("Créditos con al menos un crédito previo:", int((d["n_creditos_previos"] > 0).sum()))
print("Peor mora previa ≥ 30 días:", int((d["peor_mora_previa"] >= 30).sum()))

Pares crédito–crédito previo usados: 1261
Créditos con al menos un crédito previo: 570
Peor mora previa ≥ 30 días: 100


## 4. Buró (TransUnion) y PILA

Reutilizo, en modo de solo lectura, las variables de buró y PILA que la v1 extrajo de `bd_consultas` (fase 2D). Las acepto solo si la consulta se hizo **antes del desembolso**. Una consulta posterior no pudo informar la decisión. Los códigos especiales de CreditVision (código ≠ 0) se tratan como faltantes.

In [6]:
b = io_datos.leer_parquet(config.BURO_PILA_V0)
b["cid"] = llave(b["cid"])
b = b.set_index("cid")
f_cons = pd.to_datetime(b["f_consulta_buro"])
ok_buro = f_cons <= base["t_ref"].reindex(b.index)
print(f"Créditos con consulta de buró: {b['tiene_buro'].sum():.0f} | consulta posterior al desembolso (descartadas): "
      f"{int((b['tiene_buro'].astype(bool) & ~ok_buro).sum())}")


def valido(v):
    cod = b.get(f"{v}_cod", pd.Series(0, index=b.index)).fillna(0)
    return b[v].where(cod == 0)


B = pd.DataFrame({
    "score_buro": b["bur_score"].where(b.get("bur_score_cod", 0) == 0) if "bur_score_cod" in b else b["bur_score"],
    "sin_historial_buro": 1 - b["bur_hit"], "n_obligaciones_buro": b["bur_cantidadDeProductos"],
    "aperturas_6m": valido("cv_at06S"), "veces_30mas_12m": valido("cv_g001B"), "veces_90mas_12m": valido("cv_g003S"),
    "castigos_buro": valido("cv_co01S"), "valor_en_mora_buro": valido("cv_to_051"),
    "cuota_total_buro": b["bur_cuotaTotal"] * 1000, "salario_declarado": b["salario_declarado"],
    "pila_meses_cotizados_12m": b["pila_meses_12m"], "pila_antiguedad_aportante_meses": b["pila_meses_aportante_actual"],
    "pila_salario_sobre_declarado": b["pila_salario_sobre_declarado"]}, index=b.index)
B = B.where(ok_buro, axis=0)
d = d.join(B)
d["tiene_buro"] = d["score_buro"].notna().astype(int) | d["sin_historial_buro"].notna().astype(int)
d["cuota_sobre_salario"] = d["cuota_mensual"] / d["salario_declarado"].where(d["salario_declarado"] > 0)
d["carga_financiera_buro"] = d["cuota_total_buro"] / d["salario_declarado"].where(d["salario_declarado"] > 0)
print(f"Créditos con buró válido en el dataset: {int(d['tiene_buro'].sum())}")

Créditos con consulta de buró: 263 | consulta posterior al desembolso (descartadas): 38
Créditos con buró válido en el dataset: 225


**Lectura.** 38 de las 263 consultas de buró (14 %) se hicieron **después** del desembolso (a lo sumo 9 días después de t0), cuando el crédito ya estaba girado. La v1 las usaba como si fueran información de originación. Es una fuga pequeña pero real, y aquí queda corregida.

## 5. Exclusión del género y verificación de columnas prohibidas

El género no se lee en este notebook, así que no puede entrar al dataset. Lo dejo verificado de forma explícita, junto con el resto de variables prohibidas por la regla 6 de `AGENTS.md` (identidad y listas restrictivas).

In [7]:
prohibidas = config.PROHIBIDAS_MODELO | config.COLUMNAS_PII
assert not (set(d.columns) & prohibidas), set(d.columns) & prohibidas
print("Verificado: el dataset no contiene género, identidad, listas restrictivas ni datos personales.")

Verificado: el dataset no contiene género, identidad, listas restrictivas ni datos personales.


## 6. Ensamble final y diccionario

In [8]:
claves = ["grupo_cliente", "producto", "anio_cohorte", "conjunto_temporal", "t0", "t_ref", "elegible",
          "motivo_exclusion", "cierre_v2", "madura_6m", "madura_12m", "y_90_12m", "y_60_12m", "y_30_6m",
          "y_90_12m_B", "evento_90", "tiempo_90", "dias_hasta_30", "dias_hasta_90", "dias_obs_A"]
ds = o[claves].join(d)
ds["trimestre_t0"] = ds["t0"].dt.to_period("Q").astype(str)
ds["marcado_verificar"] = ds.index.isin(config.CREDITOS_VERIFICAR).astype(int)
ds = ds.reset_index()
ruta = io_datos.guardar_privado(ds, "dataset_analitico_v2")
print(f"Dataset analítico: {ds.shape[0]} créditos × {ds.shape[1]} columnas → {ruta.name}")

Dataset analítico: 1161 créditos × 63 columnas → dataset_analitico_v2.parquet


In [9]:
DIC = [
    ("monto", "Crédito", "num", "Valor total del crédito (COP)", "?"),
    ("n_cuotas", "Crédito", "num", "Número de cuotas pactadas", "+"),
    ("plazo_meses", "Crédito", "num", "Plazo según el plan de pagos (meses)", "+"),
    ("cuota_mensual", "Crédito", "num", "Cuota mensual equivalente (COP)", "?"),
    ("tasa_mensual", "Crédito", "num", "Tasa pactada (% mensual)", "+"),
    ("recaudo", "Crédito", "cat", "Modalidad de recaudo: empresarial (nómina), mensual o quincenal (pago directo)", "directo +"),
    ("tiene_aval", "Crédito", "bin", "Tiene aval o fianza", "?"),
    ("aval_pct", "Crédito", "num", "Porcentaje de aval", "?"),
    ("cobra_seguro", "Crédito", "bin", "Cobra seguro de vida", "?"),
    ("cobra_estudio", "Crédito", "bin", "Cobra estudio de crédito", "?"),
    ("cobra_4x1000", "Crédito", "bin", "Traslada el 4x1000 al cliente", "?"),
    ("con_prenda", "Crédito", "bin", "Libranza con prenda", "−"),
    ("tipo_solicitud", "Crédito", "cat", "Crédito nuevo, refinanciación o sin solicitud enlazada", "?"),
    ("es_refinanciacion", "Crédito", "bin", "La solicitud es una refinanciación", "+"),
    ("monto_sobre_solicitado", "Crédito", "num", "Monto aprobado / monto solicitado", "−"),
    ("edad_t0", "Cliente", "num", "Edad en la fecha de solicitud (años)", "−"),
    ("n_creditos_previos", "Historia", "num", "Créditos previos del mismo cliente", "−"),
    ("n_activos_t0", "Historia", "num", "Créditos previos aún activos en t0", "+"),
    ("es_recurrente", "Historia", "bin", "Tuvo al menos un crédito previo", "−"),
    ("peor_mora_previa", "Historia", "num", "Peor DPD de sus créditos previos hasta el día anterior a t0", "+"),
    ("meses_desde_primer_credito", "Historia", "num", "Antigüedad como cliente (meses)", "−"),
    ("cierre_previo", "Historia", "cat", "Cómo terminó su último crédito cerrado antes de t0", "con_saldo +"),
    ("emp_creditos_previos", "Empleador", "num", "Créditos originados en el mismo empleador antes de t0", "−"),
    ("sin_empleador", "Empleador", "bin", "Crédito sin empleador registrado", "+"),
    ("score_buro", "Buró", "num", "Puntaje TransUnion", "−"),
    ("sin_historial_buro", "Buró", "bin", "Sin historial en buró", "+"),
    ("n_obligaciones_buro", "Buró", "num", "Obligaciones reportadas", "?"),
    ("aperturas_6m", "Buró", "num", "Aperturas en los últimos 6 meses", "+"),
    ("veces_30mas_12m", "Buró", "num", "Veces en mora 30+ en 12 meses", "+"),
    ("veces_90mas_12m", "Buró", "num", "Veces en mora 90+ en 12 meses", "+"),
    ("castigos_buro", "Buró", "num", "Obligaciones castigadas", "+"),
    ("valor_en_mora_buro", "Buró", "num", "Valor en mora reportado (miles COP)", "+"),
    ("cuota_total_buro", "Buró", "num", "Cuota total de obligaciones (COP)", "+"),
    ("salario_declarado", "Buró", "num", "Salario declarado (COP)", "−"),
    ("cuota_sobre_salario", "Buró", "num", "Cuota del crédito / salario", "+"),
    ("carga_financiera_buro", "Buró", "num", "Cuota total en buró / salario", "+"),
    ("pila_meses_cotizados_12m", "PILA", "num", "Meses cotizados en los últimos 12", "−"),
    ("pila_antiguedad_aportante_meses", "PILA", "num", "Meses con el aportante actual", "−"),
    ("pila_salario_sobre_declarado", "PILA", "num", "Salario PILA / salario declarado", "−"),
]
dic = pd.DataFrame(DIC, columns=["variable", "grupo", "tipo", "descripcion", "signo_esperado"])
lib_el = ds[(ds["producto"] == "libranza") & ds["elegible"]]
for nombre, sub in [("libranza_elegible", lib_el), ("elegible_<2025", lib_el[lib_el["conjunto_temporal"] == "<2025"]),
                    ("elegible_2025+", lib_el[lib_el["conjunto_temporal"] == "2025+"]),
                    ("libranza_todos", ds[ds["producto"] == "libranza"])]:
    dic[f"%_disponible_{nombre}"] = [round(100 * sub[v].notna().mean(), 1) for v in dic["variable"]]
io_datos.guardar_resultado(dic, "fase1", "diccionario_v2")
dic[["variable", "grupo", "signo_esperado", "%_disponible_elegible_<2025", "%_disponible_elegible_2025+",
     "%_disponible_libranza_todos"]]

,variable,grupo,signo_esperado,%_disponible_elegible_<2025,%_disponible_elegible_2025+,%_disponible_libranza_todos
0,monto,Crédito,?,100.0,100.0,100.0
1,n_cuotas,Crédito,+,100.0,100.0,100.0
2,plazo_meses,Crédito,+,100.0,100.0,100.0
3,cuota_mensual,Crédito,?,99.8,100.0,99.9
4,tasa_mensual,Crédito,+,100.0,100.0,100.0
5,recaudo,Crédito,directo +,100.0,100.0,100.0
6,tiene_aval,Crédito,?,100.0,100.0,100.0
7,aval_pct,Crédito,?,17.2,99.2,58.6
8,cobra_seguro,Crédito,?,100.0,100.0,100.0
9,cobra_estudio,Crédito,?,100.0,100.0,100.0


**Lectura.** El diccionario deja en evidencia el problema estructural de la base. Las variables de buró y PILA no existen en los créditos elegibles de cohortes < 2025, y en 2025+ cubren solo una fracción pequeña. Están casi completas únicamente en los créditos más recientes, que aún no tienen desenlace. Con los datos propios, el modelo principal solo puede apoyarse en los términos del crédito, la historia interna y el empleador. El buró solo podrá entrar vía información externa (Home Credit como *prior*) o en el análisis de supervivencia, donde los créditos recientes aportan exposición.

El signo esperado es mi hipótesis de dominio **antes** de mirar la asociación con el evento. En el notebook 06 contrasto los signos observados contra estos.

In [10]:
_ = io_datos.escribir_registro("03_dataset_analitico", {
    "filas": len(ds), "columnas": ds.shape[1], "predictores_diccionario": len(dic),
    "libranza_elegibles": len(lib_el), "eventos_90_12m": int(lib_el["y_90_12m"].sum()),
    "excluidas_por_principio": ["genero", "identidad", "LAFT/SARLAFT", "geografía", "huella digital"],
    "salida_privada": {"archivo": "_privado/dataset_analitico_v2.parquet", "sha256": io_datos.sha256(ruta)}})

## Conclusiones

- El dataset analítico tiene una fila por crédito no anulado con el objetivo (binario y de supervivencia), las marcas de elegibilidad y 39 predictores candidatos construidos **solo con información anterior a t0**.
- Quedan vetadas por fuga `estado`, `categoria_credito`, `en_refinanciacion` y todas las tablas de desempeño. El género, la geografía, la identidad y la huella digital quedan fuera por principio.
- La historia del cliente ahora usa la mora observada hasta t0 en todos los créditos previos, y el grupo de cliente unificado.
- Se descartan 38 consultas de buró posteriores al desembolso, que la v1 usaba como información de originación.
- El buró y la PILA casi no existen en los créditos con desenlace: es la restricción de diseño más importante para la Fase 3.